In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

import os

root_dir = "/content/gdrive/MyDrive/patten_project"

# Checking if our specified directory exists
os.path.exists(root_dir)

Mounted at /content/gdrive


True

In [ ]:
import pandas as pd

# Paths to the downloaded files
data_path = os.path.join(root_dir, "train.csv")
test_path = os.path.join(root_dir,"test.csv")


# Load data
df = pd.read_csv(data_path)
tf = pd.read_csv(test_path)

## Data Preprocessing

In [ ]:
drop_vars = ["id", "shares", "n_non_stop_words"]
df.drop(drop_vars, axis = 1, inplace = True)

In [ ]:
for col in df.select_dtypes(include="number").columns:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)

In [ ]:
var_dr = ["weekday", "y", "data_channel"]
cols = df.columns.drop(var_dr)

for col in cols:
    if col.startswith("num"):
        df[col] = df[col].fillna(0.0)
    elif "polarity" in col or "subjectivity" in col:
      df[col] = df[col].fillna(0.0)
    else:
        df[col] = df[col].fillna(df[col].median())

Encode the Categorical Variables.

In [ ]:
df['data_channel'] = df['data_channel'].fillna('Other')
df['weekday'] = df['weekday'].fillna(df['weekday'].mode()[0])
df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday']).astype(int)
df = pd.get_dummies(df, columns=['data_channel', 'weekday'], prefix=['channel', 'day'])

### Train-Test Split


In [ ]:
from sklearn.model_selection import train_test_split

random_state = 100
shuffle = True
test_size_ratio = 0.25

train_df, test_df = train_test_split(df, test_size=test_size_ratio, random_state=random_state, shuffle=shuffle)
print(train_df.shape, test_df.shape)

(16650, 59) (5550, 59)


In [ ]:
target = "y"

X_train = train_df.drop(target, axis=1).values
y_train = train_df[target].values

X_test = test_df.drop(target, axis=1).values
y_test = test_df[target].values

target_names = [0, 1]
feature_names = train_df.drop(target, axis=1).columns

###Model Training



In [ ]:
from xgboost import XGBClassifier

from sklearn.model_selection import KFold, GridSearchCV

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
scoring = "f1"
models = {}

In [ ]:
model = XGBClassifier(
    objective='binary:logistic',
    gamma = 0.3,
    max_depth = 5,
    n_jobs=-1,
    random_state=random_state,
)

param_grid = {
    'min_child_weight':  [1, 7],
    'learning_rate':     [0.05, 0.2],
    'n_estimators':      [25, 50, 100],
    'reg_alpha':         [0, 0.1, 1],
    'reg_lambda':        [0.1, 1, 100],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True)
grid_search.fit(X_train, y_train)

models["XGBoost"] = grid_search.best_estimator_

In [ ]:
y_prob = models["XGBoost"].predict_proba(X_test)
print("Estimated probs:", y_prob[:10])

y_cls = models["XGBoost"].predict(X_test)
print("Estimated classes:", y_cls[:10])
print()

Estimated probs: [[0.78923327 0.21076672]
 [0.24840981 0.7515902 ]
 [0.29253775 0.70746225]
 [0.7953455  0.2046545 ]
 [0.4473564  0.5526436 ]
 [0.8589455  0.14105451]
 [0.45504898 0.544951  ]
 [0.53326905 0.46673098]
 [0.43143046 0.56856954]
 [0.11415803 0.88584197]]
Estimated classes: [0 1 1 0 1 0 1 0 1 1]



In [ ]:
threshold = 0.414
y_cls = (y_prob[:, 1] >= threshold).astype(int)
print("Estimated classes:", y_cls[:10])

Estimated classes: [0 1 1 0 1 0 1 1 1 1]


In [ ]:
f1  = f1_score(y_test, y_cls)
roc = roc_auc_score(y_test, y_prob[:, 1])
acc = accuracy_score(y_test, y_cls)

overall = (f1 + roc + acc) / 3

print(f"F1-score:    {f1:.4f}")
print(f"ROC AUC:     {roc:.4f}")
print(f"Accuracy:    {acc:.4f}")
print(f"Overall:     {overall:.4f}")

F1-score:    0.7001
ROC AUC:     0.7243
Accuracy:    0.6587
Overall:     0.6944


In [ ]:
from typing_extensions import TypeVarTuple
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler

# 1. 데이터 로드
train_df = pd.read_csv(data_path)
test_df = pd.read_csv(test_path)

# 2. 전처리 함수 정의
def preprocessing(df, drop_na=True):
    df = df.copy()

    drop_vars = ["n_non_stop_words"]
    df.drop(columns=[c for c in drop_vars if c in df.columns], inplace=True)

    # 2) 수치형 컬럼 극단치 clipping
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)
        df[col] = df[col].clip(lower, upper)

    # 3) 결측치 처리
    #    'weekday', 'y', 'data_channel' 컬럼은 별도 처리하기 위해 제외
    var_dr = {"weekday", "y", "data_channel"}
    fill_cols = [c for c in df.columns if c not in var_dr]
    for col in fill_cols:
        if col.startswith("num"):
            df[col] = df[col].fillna(0.0)
        elif "polarity" in col or "subjectivity" in col:
            df[col] = df[col].fillna(0.0)
        else:
            df[col] = df[col].fillna(df[col].median())

    df['data_channel'] = df['data_channel'].fillna('Other')
    df['weekday'] = df['weekday'].fillna(df['weekday'].mode()[0])
    df['is_weekend'] = df['weekday'].isin(['Saturday', 'Sunday']).astype(int)
    df = pd.get_dummies(df, columns=['data_channel', 'weekday'], prefix=['channel', 'day'])
    df = df.drop(columns=['id', 'url', 'timedelta', 'shares'], errors='ignore')

    if drop_na:
        df = df.dropna()

    return df

# 3. 전처리 적용
train_clean = preprocessing(train_df, drop_na=True)
test_clean = preprocessing(test_df, drop_na=False)

# 4. X, y 분리
X_train = train_clean.drop(columns=['y'])
y_train = train_clean['y']


# 5. 스케일링
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(test_clean[X_train.columns]), columns=X_train.columns)

# 6. 최적 모델 학습
model = XGBClassifier(
    objective='binary:logistic',
    gamma = 0.3,
    max_depth = 5,
    n_jobs=-1,
    random_state=random_state,
    learning_rate = 0.2,
    min_child_weight = 7,
    n_estimators = 50,
    reg_alpha = 0,
    reg_lambda = 100
)
model.fit(X_train_scaled, y_train)

# 7. 예측 (threshold 0.414 적용)
y_prob_test = model.predict_proba(X_test_scaled)[:, 1]
y_pred_test = (y_prob_test >= 0.414).astype(int)

# 8. 결과 저장
test_result = test_df.copy()
test_result['y_predict'] = y_pred_test
test_result['y_prob'] = y_prob_test
test_result.to_csv('prediction.csv', index=False)

# 9. 다운로드
from google.colab import files
files.download('prediction.csv')


<ipython-input-44-898cd4d926c3>:23: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].clip(lower, upper)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>